In [1]:
import pandas as pd
from openpyxl import load_workbook
import openpyxl
from datetime import datetime

In [2]:
#File path
schedule_master_path=r"C:\\Users\\ER128JT\\EY\\BCM Operations - Operations\\Test\\ScheduleMaster 2025\\Schedule Master_WE07022025.xlsx"
high_bench_sheet_path=r"C:\\Users\\ER128JT\\EY\\BCM Operations - Operations\\Test\\High Bench + Freshers Status Jan 27.xlsx"
book=load_workbook(high_bench_sheet_path)
sheet=book['YEA24_Master']

c:\Users\ER128JT\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\packaging\custom.py:203: UserWarning: Unknown type for FileName
  warn(f"Unknown type for {prop.name}")


In [3]:
#Load files
schedule_master=pd.read_excel(schedule_master_path,sheet_name="Master Data")
high_bench_sheet=pd.read_excel(high_bench_sheet_path,sheet_name="YEA24_Master",header=1)

c:\Users\ER128JT\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\packaging\custom.py:203: UserWarning: Unknown type for FileName
  warn(f"Unknown type for {prop.name}")


In [4]:
#Ensure necessary columns exist
required_columns_schedule=['Sector','Engagement Name']
required_columns_bench=['Current Engagement Name','Current Start Date','Current End Date','Engagement Tag']

missing_schedule=[col for col in required_columns_schedule if col not in schedule_master.columns]
missing_bench=[col for col in required_columns_bench if col not in high_bench_sheet.columns]

if missing_schedule or missing_bench:
    print(f"Missing columns! Schedule Master: {missing_schedule}, High Bench Sheet: {missing_bench}")
    #exit()

In [5]:
#Filter Schedule master for BCM sector
bcm_engagements=schedule_master[schedule_master['Sector'].str.strip().str.upper()=='BCM']['Engagement Name'].str.strip().str.upper().unique()

In [6]:
current_date = datetime.today().date()

In [7]:
#Convert Date columns to Datetime format
high_bench_sheet['Current Start Date']=pd.to_datetime(high_bench_sheet['Current Start Date'],errors='coerce').dt.date
high_bench_sheet['Current End Date']=pd.to_datetime(high_bench_sheet['Current End Date'],errors='coerce').dt.date

In [8]:
#Update Engagement Tag in High Bench Sheet
def assign_bcm_tag(row):
    if pd.notna(row['Current Engagement Name']):
        engagement_name=row['Current Engagement Name'].strip().upper()
        if engagement_name in bcm_engagements:
            if pd.notna(row['Current Start Date']) and pd.notna(row['Current End Date']):
                if row['Current Start Date']<=current_date<=row['Current End Date']:
                    return "BCM"
    return row.get("Engagement Tag","") # Retains existing value if not updated


In [9]:
#Apply function to update Engagement Tag

high_bench_sheet['Engagement Tag']=high_bench_sheet.apply(assign_bcm_tag,axis=1)

In [10]:
for idx,row in high_bench_sheet.iterrows():
    excel_row=idx+3
    sheet[f'Q{excel_row}'].value=row['Engagement Tag']

book.save(high_bench_sheet_path)

print('Updates are done')

Updates are done
